# 파주시 H3 해상도 7·8·9 테스트

파주시(운정·파주) 대상 광역버스의 정류장 좌표를 H3 해상도 7·8·9에 각각 배정하고, 셀별 정류장·노선 분포를 화면에서 비교합니다.

이 노트북은 테스트용이며 결과 파일을 저장하지 않습니다.

In [4]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

try:
    import h3
except ImportError as e:
    raise ImportError("h3 패키지가 필요합니다. VS Code 터미널에서 pip install h3 실행 후 다시 실행하세요.") from e

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "gtx_a_seoul_bus_outputs"
CACHE_DIR = BASE_DIR / "api_cache_gtx_a_routes"
CANDIDATE_FILE = OUTPUT_DIR / "gtx_a_seoul_metropolitan_bus_candidates.csv"

print("작업 폴더:", BASE_DIR)
print("캐시 폴더 존재 여부:", CACHE_DIR.exists())

작업 폴더: c:\Users\금경훈\Desktop\Gachon\3-2\UROP
캐시 폴더 존재 여부: True


In [5]:
# 파주시(운정·파주) 대상 노선 ID 추출
candidates = pd.read_csv(CANDIDATE_FILE, dtype=str, encoding="utf-8-sig")
paju_routes = candidates[
    candidates["GTX_A_역"].eq("운정중앙역")
    | candidates["대상지역"].fillna("").str.contains("파주")
].copy()
paju_route_ids = set(paju_routes["노선ID"].dropna().astype(str).str.strip())
paju_route_numbers = sorted(paju_routes["버스번호"].dropna().astype(str).str.strip().str.lstrip("'").unique())

print("파주시 대상 노선 ID 수:", len(paju_route_ids))
print("파주시 대상 버스번호:", paju_route_numbers)
display(paju_routes[["GTX_A_역", "대상지역", "버스번호", "노선ID"]].drop_duplicates())

파주시 대상 노선 ID 수: 21
파주시 대상 버스번호: ['1500', '1500(예약)', '2200', '2200-1', '3100', '3100N', '3400', '7101', '7111', '7200', '9030', '9030-1', '9709', '9709N', '9710', '9710-1', 'G7426', 'G7625', 'M7111', 'M7111(예약)', 'M7154']


,GTX_A_역,대상지역,버스번호,노선ID
136,운정중앙역,운정·파주,'1500,GGB218000010
137,운정중앙역,운정·파주,'1500(예약),GGB218000174
138,운정중앙역,운정·파주,'2200,GGB229000288
139,운정중앙역,운정·파주,'2200-1,GGB229000316
140,운정중앙역,운정·파주,'3100,GGB229000295
141,운정중앙역,운정·파주,'3100N,GGB229000294
142,운정중앙역,운정·파주,'3400,GGB218000117
143,운정중앙역,운정·파주,'7101,GGB229000273
144,운정중앙역,운정·파주,'7111,GGB229000314
145,운정중앙역,운정·파주,'7200,GGB229000072


In [6]:
# 기존 API 캐시에서 파주시 대상 노선의 정류장 좌표 읽기
stop_records = []
cache_files = list(CACHE_DIR.glob("*.json"))

for cache_file in cache_files:
    try:
        payload = json.loads(cache_file.read_text(encoding="utf-8"))
        response = payload.get("response", payload.get("Response", payload))
        body = response.get("body", {}) or {}
        items = body.get("items", {}) or {}
        items = items.get("item", []) if isinstance(items, dict) else items
        if isinstance(items, dict):
            items = [items]
        for item in items or []:
            route_id = str(item.get("routeid", "")).strip()
            lat = item.get("gpslati")
            lng = item.get("gpslong")
            if route_id in paju_route_ids and lat is not None and lng is not None:
                stop_records.append({
                    "route_id": route_id,
                    "stop_id": str(item.get("nodeid", "")).strip(),
                    "stop_no": item.get("nodeno"),
                    "latitude": float(lat),
                    "longitude": float(lng),
                    "stop_order": item.get("nodeord"),
                })
    except Exception:
        continue

stops = pd.DataFrame(stop_records).drop_duplicates(subset=["route_id", "stop_id"])
print("읽은 캐시 파일 수:", len(cache_files))
print("파주시 대상 노선 정류장 기록 수:", len(stops))
print("고유 정류장 수:", stops["stop_id"].nunique() if not stops.empty else 0)
display(stops.head())

읽은 캐시 파일 수: 970
파주시 대상 노선 정류장 기록 수: 1747
고유 정류장 수: 590


,route_id,stop_id,stop_no,latitude,longitude,stop_order
0,GGB218000174,GGB277104829,NaN,37.755133,126.738100,1
1,GGB218000174,GGB277102468,NaN,37.750867,126.739550,2
2,GGB218000174,GGB229001669,30152.0,37.748817,126.736150,3
3,GGB218000174,GGB277103813,NaN,37.744483,126.728567,4
4,GGB218000174,GGB229000852,30682.0,37.735717,126.718717,5


In [7]:
def to_h3_cell(latitude, longitude, resolution):
    # h3-py 최신 버전과 구버전 모두 지원
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(latitude, longitude, resolution)
    return h3.geo_to_h3(latitude, longitude, resolution)

if stops.empty:
    print("정류장 좌표를 읽지 못했습니다. 노선ID 체계와 캐시 파일을 확인하세요.")
else:
    comparison = []
    for resolution in [7, 8, 9]:
        temp = stops.copy()
        temp["h3_cell"] = temp.apply(
            lambda row: to_h3_cell(row["latitude"], row["longitude"], resolution),
            axis=1,
        )
        cell_summary = (
            temp.groupby("h3_cell", as_index=False)
            .agg(
                정류장수=("stop_id", "nunique"),
                노선수=("route_id", "nunique"),
                평균위도=("latitude", "mean"),
                평균경도=("longitude", "mean"),
            )
        )
        comparison.append({
            "H3해상도": resolution,
            "정류장포함셀수": len(cell_summary),
            "셀당평균정류장수": cell_summary["정류장수"].mean(),
            "셀당최대정류장수": cell_summary["정류장수"].max(),
            "셀당평균노선수": cell_summary["노선수"].mean(),
            "셀당최대노선수": cell_summary["노선수"].max(),
        })
        print(f"\n[H3 해상도 {resolution}]")
        display(cell_summary.sort_values(["정류장수", "노선수"], ascending=False).head(20))

    comparison_df = pd.DataFrame(comparison)
    print("해상도 비교")
    display(comparison_df)
    print("참고: 이 테스트는 화면 표시만 하며 CSV·지도 파일을 저장하지 않습니다.")


[H3 해상도 7]


,h3_cell,정류장수,노선수,평균위도,평균경도
21,8730e0b48ffffff,24,13,37.726751,126.752874
90,8730e578bffffff,19,2,37.856575,126.786626
17,8730e0b41ffffff,18,11,37.728513,126.718256
26,8730e0b4effffff,18,8,37.719317,126.738082
85,8730e56b2ffffff,17,3,37.757002,126.778091
23,8730e0b4affffff,16,13,37.710578,126.750572
55,8730e1d8cffffff,15,10,37.566937,126.976196
24,8730e0b4cffffff,14,11,37.734797,126.733946
19,8730e0b43ffffff,14,2,37.715290,126.714738
11,8730e0a69ffffff,13,4,37.691514,126.862046



[H3 해상도 8]


,h3_cell,정류장수,노선수,평균위도,평균경도
222,8830e578bbfffff,9,2,37.859429,126.787059
47,8830e0b481fffff,7,9,37.725680,126.752238
17,8830e0a697fffff,7,4,37.687914,126.865321
130,8830e1d869fffff,7,4,37.552218,126.917486
91,8830e0b6b7fffff,6,10,37.751825,126.741027
132,8830e1d8c1fffff,6,10,37.569883,126.977312
51,8830e0b48bfffff,6,7,37.731954,126.758187
67,8830e0b4e1fffff,6,7,37.719198,126.736915
181,8830e56955fffff,6,4,37.743633,126.807411
29,8830e0b405fffff,6,3,37.716789,126.695404



[H3 해상도 9]


,h3_cell,정류장수,노선수,평균위도,평균경도
144,8930e0b6b63ffff,5,6,37.752464,126.742011
101,8930e0b4c9bffff,4,10,37.744135,126.728661
214,8930e1d8c13ffff,4,10,37.569827,126.976615
349,8930e57882fffff,4,1,37.862796,126.774538
11,8930e0a5b5bffff,3,10,37.575111,126.865214
112,8930e0b4e0fffff,3,5,37.717190,126.737148
274,8930e1db347ffff,3,5,37.533107,126.903503
23,8930e0a6963ffff,3,4,37.687583,126.865567
137,8930e0b6953ffff,3,4,37.772504,126.733267
179,8930e1d32c3ffff,3,4,37.668283,126.888444


해상도 비교


,H3해상도,정류장포함셀수,셀당평균정류장수,셀당최대정류장수,셀당평균노선수,셀당최대노선수
0,7,92,6.413043,24,4.956522,17
1,8,225,2.622222,9,3.595556,15
2,9,369,1.598916,5,3.089431,10


참고: 이 테스트는 화면 표시만 하며 CSV·지도 파일을 저장하지 않습니다.


In [8]:
# H3 셀과 정류장 위치를 지도에 표시합니다. 지도 파일은 저장하지 않습니다.
try:
    import folium
except ImportError as e:
    raise ImportError("folium 패키지가 필요합니다. VS Code 터미널에서 pip install folium 실행 후 다시 실행하세요.") from e

if stops.empty:
    print("표시할 정류장 좌표가 없습니다. 앞 셀부터 순서대로 실행하세요.")
else:
    center = [stops["latitude"].mean(), stops["longitude"].mean()]

    for resolution, color in [(7, "red"), (8, "blue"), (9, "green")]:
        map_df = stops.copy()
        map_df["h3_cell"] = map_df.apply(
            lambda row: to_h3_cell(row["latitude"], row["longitude"], resolution),
            axis=1,
        )
        m = folium.Map(location=center, zoom_start=11, tiles="CartoDB positron")

        for cell in map_df["h3_cell"].unique():
            if hasattr(h3, "cell_to_boundary"):
                boundary = h3.cell_to_boundary(cell)
            else:
                boundary = h3.h3_to_geo_boundary(cell)
            folium.Polygon(
                locations=boundary,
                color=color,
                weight=2,
                fill=True,
                fill_color=color,
                fill_opacity=0.12,
                tooltip=f"H3 {resolution}: {cell}",
            ).add_to(m)

        for _, row in map_df.drop_duplicates("stop_id").iterrows():
            folium.CircleMarker(
                location=[row["latitude"], row["longitude"]],
                radius=3,
                color="black",
                fill=True,
                fill_color="yellow",
                fill_opacity=0.9,
                tooltip=f"정류장 ID: {row['stop_id']}",
            ).add_to(m)

        print(f"H3 해상도 {resolution} 지도")
        display(m)

H3 해상도 7 지도


H3 해상도 8 지도


H3 해상도 9 지도
